# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [13]:
# Task 1
# YOUR CODE HERE
# Step 1: Keep fossil sources only
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# Step 2: Find the biggest fossil users (for the title)
by_region = df_fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False)
by_country = df_fossil.groupby('Country')['TWh'].sum().sort_values(ascending=False)
top_region, top_country = by_region.index[0], by_country.index[0]
top_source = (df_fossil[df_fossil['Country'] == top_country]
              .sort_values('TWh', ascending=False)['Source'].iloc[0])   # its biggest fossil source

# Step 3: CVD-safe colours for the three fossil sources (indigo / amber / mint)
fossil_colors = {'Coal': '#4338A8', 'Oil': '#E6A016', 'Natural Gas': '#79DDB8'}

# Step 4: Treemap Region -> Country -> Source
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    height=700,
    template='plotly_white'
)

# Step 5: TWh labels only (no percentages)
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white'
)

# Step 6: Use light grey for region and country tiles
trace = fig.data[0]
parent_ids = set(trace.parents)
trace.marker.colors = [
    '#DDDDDD' if node_id in parent_ids else color
    for node_id, color in zip(trace.ids, trace.marker.colors)
]

# Step 7: Insight title
fig.update_layout(
    title=(f"{top_region} burns the most fossil energy ({by_region.iloc[0]:,.0f} TWh) — "
           f"but {top_country} is the single biggest user ({by_country.iloc[0]:,.0f} TWh, mostly {top_source.lower()})"),
    font=dict(family='Arial', size=12),
    margin=dict(t=60, l=10, r=10, b=10)
)

fig.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:
# Task 2 — charts
# YOUR CODE HERE

# Step 1: Low-carbon only, total TWh per country
df_low = (df.loc[df['Source_Type'] == 'Low-carbon']
            .groupby('Country', as_index=False)['TWh'].sum()
            .sort_values('TWh', ascending=False))
df_low['All'] = 'Low-carbon'          # dummy root node
leader = df_low.iloc[0]

# Step 2: Treemap — single level under one root
fig = px.treemap(
    df_low,
    path=['All', 'Country'],
    values='TWh',
    color_discrete_sequence=['#2E75B6'],
    height=550,
    template='plotly_white'
)
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white'
)
fig.update_layout(title='Low-carbon energy (nuclear + hydro) by country — treemap',
                  font=dict(family='Arial', size=12), margin=dict(t=60, l=10, r=10, b=10))
fig.show()

# Step 3: Horizontal bar — sorted, one CVD-safe blue, TWh labels
fig = px.bar(
    df_low.sort_values('TWh'),        # ascending -> largest ends on top
    x='TWh', y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=['#2E75B6'],
    labels={'TWh': 'Low-carbon energy (TWh)', 'Country': ''},
    height=550,
    template='plotly_white'
)
fig.update_traces(texttemplate='%{text:,.0f} TWh', textposition='outside', cliponaxis=False)
fig.update_layout(
    title=f"{leader['Country']} leads low-carbon energy with {leader['TWh']:,.0f} TWh of nuclear + hydro",
    font=dict(family='Arial', size=12),
    xaxis=dict(range=[0, df_low['TWh'].max() * 1.15], showgrid=False),
    bargap=0.25
)
fig.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*YOUR ANSWER HERE*

The bar chart, clearly. All the bars start from the same line, so you just compare lengths and the
order jumps out straight away: France first, then Norway, then Canada.

With the treemap I had to guess which box was bigger. France, Norway and Canada look almost the same
size, and the small countries are hard to read at all. The treemap is nice for seeing how the total is
split up, but for ranking, the bar chart wins.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [4]:
# Task 3
# YOUR CODE HERE

# Step 1: Region x Source totals, then share of each region's fossil total
mix = df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum()
mix['Share'] = mix['TWh'] / mix.groupby('Region')['TWh'].transform('sum')

# Step 2: Sort regions by coal share (highest at the top)
coal = mix.loc[mix['Source'] == 'Coal'].sort_values('Share', ascending=False)
region_order = coal['Region'].tolist() + [r for r in mix['Region'].unique()
                                          if r not in coal['Region'].tolist()]   # Middle East has no coal -> bottom
top = coal.iloc[0]

# Step 3: Labels on coal segments only
mix['Label'] = np.where(mix['Source'] == 'Coal', mix['Share'].map('{:.0%}'.format), '')

# Step 4: Coal = same orange as Task 1; Oil and Gas in two greys
accent = {'Coal': '#E07B39', 'Oil': '#BBBBBB', 'Natural Gas': '#DDDDDD'}

fig = px.bar(
    mix,
    x='Share', y='Region',
    orientation='h',
    color='Source',
    text='Label',
    color_discrete_map=accent,
    category_orders={'Region': region_order,
                     'Source': ['Coal', 'Oil', 'Natural Gas']},   # coal on the baseline
    hover_data={'Label': False, 'Share': ':.0%', 'TWh': ':,.0f'},
    labels={'Share': 'Share of fossil energy', 'Region': '', 'Source': ''},
    height=500,
    template='plotly_white'
)

# Step 5: Customisation
fig.update_traces(textposition='inside', insidetextanchor='start',
                  textfont=dict(color='white', size=12),
                  marker_line_color='white', marker_line_width=1)
fig.update_layout(
    title=f"{top['Region']} relies most on coal — {top['Share']:.0%} of its fossil energy",
    font=dict(family='Arial', size=12),
    xaxis=dict(tickformat='.0%', range=[0, 1]),
    legend=dict(orientation='h', x=0, y=1.08),
    bargap=0.25
)

fig.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

*YOUR ANSWER HERE*

The bar chart is clearer. The bars are sorted and start from the same line, so I can see the ranking
straight away: France, then Norway, then Canada.

In the treemap it's hard to tell which box is bigger, especially France, Norway and Canada, which look
almost the same. The treemap is better for showing the total, but the bar chart is better for ranking.